# Kaggle 01 — COBOL Corpus Ingest

**Week 2 task** — Run on Kaggle CPU (32 cores, niente GPU necessaria).

Ingest pipeline (priorità):
1. The Stack v2 dedup COBOL (~36k file) — content scaricato da Software Heritage S3
2. X-COBOL Zenodo (85 repo) — dataset Kaggle `xcobol`
3. Community repos — clonati inline

(XMainframe corpus 236M NON è pubblico → distillato come teacher in W3.
 The Stack v1 è il fallback se mancano credenziali AWS.)

Output: HF Hub private dataset `AlexThunder0/cobol-cpt-corpus`

**Prerequisiti Kaggle Secrets:** `HF_TOKEN`, `GH_TOKEN`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`

In [ ]:
!pip install datasets huggingface-hub datasketch tqdm "smart_open[s3]" boto3 -q

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
GH_TOKEN = secrets.get_secret('GH_TOKEN')

# Credenziali AWS per scaricare il content di The Stack v2 da Software Heritage S3
os.environ['AWS_ACCESS_KEY_ID'] = secrets.get_secret('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = secrets.get_secret('AWS_SECRET_ACCESS_KEY')

print('Secrets configurati (HF, GH, AWS)')

In [ ]:
!sudo apt-get install -y gnucobol -q
!cobc --version

In [ ]:
import subprocess, sys, os, shutil

# Rimuovi clone precedente se esiste (warm container)
shutil.rmtree('/kaggle/working/Qwen-COBOL', ignore_errors=True)

with open(os.path.expanduser('~/.netrc'), 'w') as f:
    f.write(f'machine github.com login x-access-token password {GH_TOKEN}\n')
os.chmod(os.path.expanduser('~/.netrc'), 0o600)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--quiet',
     'https://github.com/AlexThunder01/Qwen-COBOL.git',
     '/kaggle/working/Qwen-COBOL'],
    check=True,
    env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'},
)
sys.path.insert(0, '/kaggle/working/Qwen-COBOL')
print('Repo clonato')

In [ ]:
import os
os.makedirs('/kaggle/working/community_repos', exist_ok=True)

repos = [
    'https://github.com/openmainframeproject/cobol-programming-course',
    'https://github.com/Martinfx/COBOL',
]
for url in repos:
    name = url.split('/')[-1]
    dest = f'/kaggle/working/community_repos/{name}'
    if not os.path.exists(dest):
        ret = os.system(f'git clone --depth 1 --quiet {url} {dest}')
        if ret == 0:
            print(f'Clonato: {name}')
        else:
            print(f'ERRORE clone: {name}')
    else:
        print(f'Già presente: {name}')

In [ ]:
from src.pipeline.ingest import run_ingest
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

run_ingest(
    zenodo_dir='/kaggle/input/datasets/alexander912/xcobol',
    repos_dir='/kaggle/working/community_repos',
    gnucobol_share='/usr/share/gnucobol',
    skip_stack=False,
    skip_xmainframe=True,  # Fsoft-AIC/XMainframe non esiste su HF Hub
)

In [ ]:
import os
from huggingface_hub import hf_hub_download
import pandas as pd

path = hf_hub_download(
    repo_id='AlexThunder0/cobol-cpt-corpus',
    filename='data/train-00000-of-00001.parquet',
    repo_type='dataset',
    token=os.environ['HF_TOKEN'],
)
df = pd.read_parquet(path)

total_chars = df['content'].str.len().sum()
print('=== CORPUS STATS ===')
print(f'Totale record:   {len(df):,}')
print(f'Totale chars:    {total_chars:,}')
print(f'Stima token:     ~{total_chars/4:,.0f}')
print(f'\nPer fonte:')
print(df['source'].value_counts())
print(f'\nLunghezza media: {df["content"].str.len().mean():,.0f} chars/record')
print(f'difficulty_score medio: {df["difficulty_score"].mean():.3f}')